# Домашнее задание: Реализация RAG-системы с поиском по собственной базе документов (ChromaDB)

## Описание
В данном ноутбуке мы:
1. Настроим окружение и установим необходимые библиотеки.
2. Подготовим синтетический датасет.
3. Создадим эмбеддинги с использованием `sentence-transformers`.
4. Настроим и проиндексируем данные в ChromaDB (изучим HNSW).
5. Реализуем семантический поиск, фильтрацию по метаданным.
6. Проведем анализ производительности.

In [14]:
import sys
import os
import time
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from typing import List, Dict

print(f"Python executable: {sys.executable}")
print(f"ChromaDB version: {chromadb.__version__}")

Python executable: /home/zodiac/stadygit/LLM-Driven-Development/.venv/bin/python
ChromaDB version: 1.4.1


In [15]:
# 2. Подготовка набора данных (Dataset)

documents_data = [
    {"id": "1", "text": "Искусственный интеллект трансформирует отрасли, автоматизируя процессы и улучшая принятие решений.", "category": "tech"},
    {"id": "2", "text": "Квантовые компьютеры используют принципы квантовой механики для выполнения вычислений с невероятной скоростью.", "category": "tech"},
    {"id": "3", "text": "Python — это популярный язык программирования, известный своей простотой и мощными библиотеками для анализа данных.", "category": "tech"},
    {"id": "4", "text": "Большой Барьерный риф является крупнейшей в мире системой коралловых рифов, расположенной у побережья Австралии.", "category": "nature"},
    {"id": "5", "text": "Амазонские тропические леса производят значительную часть кислорода Земли и являются домом для миллионов видов.", "category": "nature"},
    {"id": "6", "text": "Изменение климата приводит к повышению глобальной температуры и экстремальным погодным явлениям.", "category": "nature"},
    {"id": "7", "text": "Блокчейн обеспечивает децентрализованное и безопасное хранение записей о транзакциях.", "category": "tech"},
    {"id": "8", "text": "Фотосинтез — это процесс, с помощью которого зеленые растения используют солнечный свет для синтеза питательных веществ.", "category": "nature"},
    {"id": "9", "text": "Нейронные сети имитируют работу человеческого мозга для решения задач распознавания образов.", "category": "tech"},
    {"id": "10", "text": "Панды питаются почти исключительно бамбуком и обитают в горных регионах Китая.", "category": "nature"}
]

df = pd.DataFrame(documents_data)
print("Пример данных:")
display(df.head())

Пример данных:


,id,text,category
0,1,Искусственный интеллект трансформирует отрасли...,tech
1,2,Квантовые компьютеры используют принципы квант...,tech
2,3,"Python — это популярный язык программирования,...",tech
3,4,Большой Барьерный риф является крупнейшей в ми...,nature
4,5,Амазонские тропические леса производят значите...,nature


In [16]:

model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
embedding_model = SentenceTransformer(model_name)

sample_embedding = embedding_model.encode("Пример текста")
print(f"Размерность вектора: {len(sample_embedding)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 674.36it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Размерность вектора: 384


# Часть 1. Настройка и индексация (ChromaDB)

In [17]:

client = chromadb.Client()

try:
    client.delete_collection("my_rag_collection")
except Exception:

    pass

collection = client.create_collection(
    name="my_rag_collection",
    metadata={
        "hnsw:space": "cosine", 
        "hnsw:construction_ef": 128,
        "hnsw:M": 24 
    }
)

print("Коллекция создана.")

Коллекция создана.


In [18]:

print("Генерация эмбеддингов...")
embeddings = embedding_model.encode(df['text'].tolist())

ids = df['id'].tolist()
documents = df['text'].tolist()
metadatas = [{"category": cat} for cat in df['category']]

collection.add(
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Добавлено {collection.count()} документов.")

Генерация эмбеддингов...
Добавлено 10 документов.


# Часть 2. Реализация поиска


In [19]:


def search(query_text: str, n_results: int = 3, category_filter: str = None):

    query_embedding = embedding_model.encode([query_text])

    where_filter = None
    if category_filter:
        where_filter = {"category": category_filter}
        
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where_filter,
        include=['documents', 'metadatas', 'distances']
    )
    
    return results

# Тестовые запросы
print("--- Запрос: 'технологии будущего' ---")
res1 = search("технологии будущего", n_results=3)
for i in range(len(res1['ids'][0])):
    print(f"Doc: {res1['documents'][0][i]}")
    print(f"Dist: {res1['distances'][0][i]:.4f} | Meta: {res1['metadatas'][0][i]}\n")
    
print("--- Запрос: 'экология и природа' (Фильтр: category='nature') ---")
res2 = search("экология и природа", n_results=3, category_filter='nature')
for i in range(len(res2['ids'][0])):
    print(f"Doc: {res2['documents'][0][i]}")
    print(f"Dist: {res2['distances'][0][i]:.4f} | Meta: {res2['metadatas'][0][i]}\n")

--- Запрос: 'технологии будущего' ---
Doc: Искусственный интеллект трансформирует отрасли, автоматизируя процессы и улучшая принятие решений.
Dist: 0.6324 | Meta: {'category': 'tech'}

Doc: Квантовые компьютеры используют принципы квантовой механики для выполнения вычислений с невероятной скоростью.
Dist: 0.7124 | Meta: {'category': 'tech'}

Doc: Нейронные сети имитируют работу человеческого мозга для решения задач распознавания образов.
Dist: 0.8152 | Meta: {'category': 'tech'}

--- Запрос: 'экология и природа' (Фильтр: category='nature') ---
Doc: Амазонские тропические леса производят значительную часть кислорода Земли и являются домом для миллионов видов.
Dist: 0.6727 | Meta: {'category': 'nature'}

Doc: Фотосинтез — это процесс, с помощью которого зеленые растения используют солнечный свет для синтеза питательных веществ.
Dist: 0.7027 | Meta: {'category': 'nature'}

Doc: Изменение климата приводит к повышению глобальной температуры и экстремальным погодным явлениям.
Dist: 0.7425 | 

In [20]:
def benchmark_search(n_queries=100, top_k=5):
    start_time = time.time()

    base_query = "анализ данных и алгоритмы"

    query_vec = embedding_model.encode([base_query])
    
    for _ in range(n_queries):
        collection.query(
            query_embeddings=query_vec,
            n_results=top_k
        )
        
    end_time = time.time()
    avg_time = (end_time - start_time) / n_queries
    print(f"Среднее время поиска (на {n_queries} запросов, top-k={top_k}): {avg_time*1000:.4f} ms")

print("Тест производительности:")
benchmark_search(n_queries=100, top_k=1)
benchmark_search(n_queries=100, top_k=5)
benchmark_search(n_queries=100, top_k=10)

Тест производительности:
Среднее время поиска (на 100 запросов, top-k=1): 1.4764 ms
Среднее время поиска (на 100 запросов, top-k=5): 1.4481 ms
Среднее время поиска (на 100 запросов, top-k=10): 1.5057 ms


In [21]:
queries = [
    "зеленые технологии",
    "история компьютеров",
    "разнообразие животного мира",
    "защита информации"
]

batch_embeddings = embedding_model.encode(queries)

start_batch = time.time()
batch_results = collection.query(
    query_embeddings=batch_embeddings,
    n_results=2
)
end_batch = time.time()

print(f"Время выполнения батча из {len(queries)} запросов: {(end_batch - start_batch)*1000:.4f} ms")

for i, q in enumerate(queries):
    print(f"\nQuery: {q}")
    for j in range(len(batch_results['ids'][i])):
        print(f" - Found: {batch_results['documents'][i][j]} (Dist: {batch_results['distances'][i][j]:.4f})")

Время выполнения батча из 4 запросов: 1.8525 ms

Query: зеленые технологии
 - Found: Фотосинтез — это процесс, с помощью которого зеленые растения используют солнечный свет для синтеза питательных веществ. (Dist: 0.4834)
 - Found: Искусственный интеллект трансформирует отрасли, автоматизируя процессы и улучшая принятие решений. (Dist: 0.7069)

Query: история компьютеров
 - Found: Нейронные сети имитируют работу человеческого мозга для решения задач распознавания образов. (Dist: 0.6750)
 - Found: Квантовые компьютеры используют принципы квантовой механики для выполнения вычислений с невероятной скоростью. (Dist: 0.6952)

Query: разнообразие животного мира
 - Found: Панды питаются почти исключительно бамбуком и обитают в горных регионах Китая. (Dist: 0.7059)
 - Found: Амазонские тропические леса производят значительную часть кислорода Земли и являются домом для миллионов видов. (Dist: 0.7449)

Query: защита информации
 - Found: Блокчейн обеспечивает децентрализованное и безопасное хранен